In [ ]:
import pandas as pd

ord_df = pd.read_csv('/content/orders.csv')

ord_df.head()

In [ ]:
users_df = pd.read_json('/content/users.json')

users_df.head()

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')

with open('/content/restaurants.sql', 'r') as f:
    sql_script = f.read()

conn.executescript(sql_script)


In [ ]:
restaurants_df = pd.read_sql_query('SELECT * FROM restaurants', conn)

restaurants_df.head()

In [ ]:
merged_df = pd.merge(ord_df, users_df, on='user_id', how='left')
merged_df = pd.merge(merged_df, restaurants_df, on='restaurant_id', how='left')

merged_df.head()

In [ ]:
merged_df.to_csv('final_food_delivery_dataset.csv', index=False)


In [ ]:

gold_members_df = merged_df[merged_df['membership'] == 'Gold']

city_revenue = gold_members_df.groupby('city')['total_amount'].sum().reset_index()

highest_revenue_city = city_revenue.loc[city_revenue['total_amount'].idxmax()]

print(f"The city with the highest total revenue from Gold members: {highest_revenue_city['city']}")

In [ ]:

cuisine_avg_order = merged_df.groupby('cuisine')['total_amount'].mean().reset_index()

highest_avg_cuisine = cuisine_avg_order.loc[cuisine_avg_order['total_amount'].idxmax()]

print(f"The cuisine with the highest average order value is: {highest_avg_cuisine['cuisine']}")

In [ ]:

user_total_spending = merged_df.groupby('user_id')['total_amount'].sum().reset_index()

users_above_1000 = user_total_spending[user_total_spending['total_amount'] > 1000]

num_distinct_users = users_above_1000['user_id'].nunique()

print(f"Number of distinct users who placed orders worth more than ₹1000 in total: {num_distinct_users}")
if num_distinct_users < 500:
    print("Range: < 500")
elif 500 <= num_distinct_users <= 1000:
    print("Range: 500 – 1000")
elif 1000 < num_distinct_users <= 2000:
    print("Range: 1000 – 2000")
else:
    print("Range: > 2000")

In [ ]:
bins = [3.0, 3.5, 4.0, 4.5, 5.0]
labels = ['3.0 – 3.5', '3.6 – 4.0', '4.1 – 4.5', '4.6 – 5.0']
merged_df['rating_range'] = pd.cut(merged_df['rating'], bins=bins, labels=labels, right=True, include_lowest=True)
rating_revenue = merged_df.groupby('rating_range', observed=False)['total_amount'].sum().reset_index()
highest_revenue_rating_range = rating_revenue.loc[rating_revenue['total_amount'].idxmax()]
print(f"The restaurant rating range that generated the highest total revenue is: {highest_revenue_rating_range['rating_range']}")

In [ ]:
gold_members_df = merged_df[merged_df['membership'] == 'Gold']
gold_city_avg_order = gold_members_df.groupby('city')['total_amount'].mean().reset_index()
highest_avg_order_gold_city = gold_city_avg_order.loc[gold_city_avg_order['total_amount'].idxmax()]
print(f"Among Gold members, the city with the highest average order value is: {highest_avg_order_gold_city['city']}")

In [ ]:
cuisine_restaurants_count = merged_df.groupby('cuisine')['restaurant_name_y'].nunique().reset_index(name='distinct_restaurants')
cuisine_total_revenue = merged_df.groupby('cuisine')['total_amount'].sum().reset_index(name='total_revenue')
cuisine_summary = pd.merge(cuisine_restaurants_count, cuisine_total_revenue, on='cuisine')
cuisine_summary_sorted = cuisine_summary.sort_values(by=['distinct_restaurants', 'total_revenue'], ascending=[True, False])
print(f"\nBased on the analysis, {cuisine_summary_sorted.iloc[0]['cuisine']}")

In [ ]:
total_orders = len(merged_df)
gold_member_orders = merged_df[merged_df['membership'] == 'Gold'].shape[0]
percentage_gold_orders = (gold_member_orders / total_orders) * 100
print(f"Percentage of total orders by Gold members: {percentage_gold_orders:.0f}%")

In [ ]:
restaurant_summary = merged_df.groupby('restaurant_name_y').agg(
    order_count=('order_id', 'count'),
    avg_total_amount=('total_amount', 'mean')
).reset_index()
filtered_restaurants = restaurant_summary[restaurant_summary['order_count'] < 20]
highest_avg_order_restaurant = filtered_restaurants.loc[filtered_restaurants['avg_total_amount'].idxmax()]
print(f"The restaurant with the highest average order value is: {highest_avg_order_restaurant['restaurant_name_y']}")

In [ ]:
combinations = [
    ('Gold', 'Indian'),
    ('Gold', 'Italian'),
    ('Regular', 'Indian'),
    ('Regular', 'Chinese')
]
revenue_by_combination = {}
for membership_type, cuisine_type in combinations:
    filtered_df = merged_df[
        (merged_df['membership'] == membership_type) &
        (merged_df['cuisine'] == cuisine_type)
    ]
    total_revenue = filtered_df['total_amount'].sum()
    revenue_by_combination[f"{membership_type} + {cuisine_type}"] = total_revenue
highest_revenue_combination = max(revenue_by_combination, key=revenue_by_combination.get)
highest_revenue_value = revenue_by_combination[highest_revenue_combination]
print("Revenue for each combination:")
for combo, revenue in revenue_by_combination.items():
    print(f"- {combo}: ₹{revenue:.2f}")
print(f"\nThe combination that contributes the highest revenue is: {highest_revenue_combination}")

In [ ]:
merged_df['order_date'] = pd.to_datetime(merged_df['order_date'], format='%d-%m-%Y')
merged_df['quarter'] = merged_df['order_date'].dt.quarter
highest_revenue_quarter = quarterly_revenue.loc[quarterly_revenue['total_amount'].idxmax()]
quarter_mapping = {
    1: 'Q1 (Jan–Mar)',
    2: 'Q2 (Apr–Jun)',
    3: 'Q3 (Jul–Sep)',
    4: 'Q4 (Oct–Dec)'
}
print(f"The quarter with the highest total revenue is: {quarter_mapping[highest_revenue_quarter['quarter']]}")